In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q10/annotated/checkpoints/pre_cell_4.pickle

In [ ]:
%%RecordEvent
%%time
### cell 4 ###

# Compute revenue early and drop unnecessary columns to reduce data shuffled
rev = (
    lineitem
    .loc[lineitem.L_RETURNFLAG == 'R', ['L_ORDERKEY', 'L_EXTENDEDPRICE', 'L_DISCOUNT']]
    .assign(TMP=lambda df: df.L_EXTENDEDPRICE * (1 - df.L_DISCOUNT))
    .drop(['L_EXTENDEDPRICE', 'L_DISCOUNT'], axis=1)
)

# Filter orders by date
ord_f = orders.loc[
    (orders.O_ORDERDATE >= date1) & (orders.O_ORDERDATE < date2),
    ['O_ORDERKEY', 'O_CUSTKEY']
]

# Pre‐select customer and nation columns
cust_sel = customer[
    ['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'C_NATIONKEY', 'C_ADDRESS', 'C_COMMENT']
]
nation_sel = nation[['N_NATIONKEY', 'N_NAME']]

# Perform joins on smaller intermediate tables
total = (
    rev
    .merge(ord_f, left_on='L_ORDERKEY', right_on='O_ORDERKEY')
    .merge(cust_sel, left_on='O_CUSTKEY', right_on='C_CUSTKEY')
    .merge(nation_sel, left_on='C_NATIONKEY', right_on='N_NATIONKEY')
    .groupby(
        ['C_CUSTKEY', 'C_NAME', 'C_ACCTBAL', 'C_PHONE', 'N_NAME', 'C_ADDRESS', 'C_COMMENT'],
        as_index=False,
        sort=False
    )['TMP']
    .sum()
    .sort_values('TMP', ascending=False)
)


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q10/rewritten/o4_mini_high/checkpoints/post_cell_4_try_3.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/dias-benchmarks/tpch/notebooks/q10/opt_cell_exec_info_4_try_3.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[4], f)


In [ ]:
opt_output = Out.get(4)